# Huấn luyện CSRNet đếm người trên ShanghaiTech Part B

**Đề tài:** Hệ thống đếm người, phát hiện đám đông bất thường phục vụ an ninh nơi công cộng.

Pipeline của notebook (khớp Chương 3–4 của báo cáo):

1. Cài đặt môi trường, tải bộ dữ liệu **ShanghaiTech Part B**
2. Khảo sát dữ liệu (EDA)
3. Tiền xử lý: sinh **bản đồ mật độ** ground-truth từ nhãn điểm đầu người
4. Chia tập train / val / test, tăng cường dữ liệu
5. Xây dựng mô hình **CSRNet** (VGG-16 + tích chập giãn)
6. Huấn luyện, chọn mô hình tốt nhất theo MAE trên tập val
7. Đánh giá trên tập test: **MAE, RMSE**, tốc độ suy luận
8. Thử luật cảnh báo mật độ trên kết quả dự đoán
9. Xuất trọng số `csrnet_partB.pth` để dùng trong app demo

> Chạy trên **Google Colab** với GPU (Runtime → Change runtime type → T4 GPU). Toàn bộ hình/kết quả được lưu vào thư mục `outputs/` để chèn vào báo cáo.

## 1. Cài đặt và cấu hình

In [ ]:
!pip -q install kagglehub

In [ ]:
import os, glob, json, random, time, shutil
import numpy as np
import cv2
import scipy.io as sio
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

CFG = dict(
    part="B",            # "B": ảnh đường phố (gần với CCTV); "A": đám đông rất dày
    sigma=15,            # độ rộng nhân Gauss cố định cho Part B (như CSRNet)
    adaptive=False,      # True: nhân Gauss thích ứng hình học (nên dùng cho Part A)
    crop=512,            # kích thước ô cắt ngẫu nhiên khi huấn luyện (bội số của 8)
    batch_size=4,
    epochs=150,
    lr=1e-5,
    weight_decay=1e-4,
    density_scale=100.0, # nhân bản đồ mật độ để gradient không quá nhỏ
    val_ratio=0.1,
    eval_every=2,
    seed=42,
)
OUT = "outputs"; os.makedirs(OUT, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["seed"])
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## 2. Tải bộ dữ liệu ShanghaiTech

Bộ dữ liệu gốc của Zhang và cộng sự (CVPR 2016). Part B gồm **716 ảnh** chụp đường phố Thượng Hải (400 train / 316 test), mỗi đầu người được gán một điểm (x, y) trong file `.mat`.

Notebook tải bản sao trên Kaggle bằng `kagglehub`. Nếu lỗi, tải thủ công file zip ShanghaiTech (Kaggle hoặc link trong repo MCNN/CSRNet), giải nén rồi gán `DATA_ROOT` tới thư mục đó — đoạn code dưới tự tìm thư mục `part_B*`.

In [ ]:
DATA_ROOT = None  # điền đường dẫn nếu đã có dữ liệu sẵn, ví dụ "/content/ShanghaiTech"
if DATA_ROOT is None:
    import kagglehub
    DATA_ROOT = kagglehub.dataset_download("tthien/shanghaitech")
print("DATA_ROOT =", DATA_ROOT)

def find_split(root, part, split):
    cands = sorted(glob.glob(os.path.join(root, "**", f"part_{part}*", f"{split}_data"), recursive=True))
    assert cands, f"Không tìm thấy part_{part}*/{split}_data trong {root}"
    return cands[0]

TRAIN_DIR = find_split(DATA_ROOT, CFG["part"], "train")
TEST_DIR = find_split(DATA_ROOT, CFG["part"], "test")

def list_items(split_dir):
    items = []
    for img in sorted(glob.glob(os.path.join(split_dir, "images", "*.jpg"))):
        name = os.path.splitext(os.path.basename(img))[0]
        gt = os.path.join(split_dir, "ground-truth", f"GT_{name}.mat")
        if os.path.exists(gt):
            items.append({"img": img, "gt": gt, "name": name})
    return items

train_all = list_items(TRAIN_DIR)
test_items = list_items(TEST_DIR)
print(len(train_all), "ảnh train,", len(test_items), "ảnh test")

## 3. Khảo sát dữ liệu (EDA)

Mục tiêu: hiểu dữ liệu trước khi huấn luyện, và dùng các quan sát để chọn tham số tiền xử lý (σ của nhân Gauss, kích thước ô cắt) và để giải thích kết quả sau này.

| Mục | Câu hỏi | Hình xuất ra |
|---|---|---|
| 3.1 | Mỗi ảnh có bao nhiêu người, train và test có giống nhau không? | bảng thống kê |
| 3.2 | Ảnh trông như thế nào từ thưa đến dày? | `gallery.png` |
| 3.3 | Phân bố số người mỗi ảnh | `hist_counts.png` |
| 3.4 | Nhãn điểm đầu người trông thế nào, vùng đông nhất ra sao? | `sample_points.png` |
| 3.5 | Người thường xuất hiện ở vùng nào trong khung hình? | `spatial_heatmap.png` |
| 3.6 | Phối cảnh: khoảng cách giữa các đầu người thay đổi theo vị trí thế nào? | `perspective_nn.png` |
| 3.7 | Mật độ cục bộ: có bao nhiêu “điểm nóng”? | `local_density.png` |
| 3.8 | Độ sáng, độ tương phản, kích thước ảnh | `image_props.png` |
| 3.9 | Dữ liệu trải qua những mức cảnh báo nào? | `level_distribution.png` |

### 3.1. Đọc nhãn và bảng thống kê tổng quát

In [ ]:
import pandas as pd

def load_points(mat_path):
    info = sio.loadmat(mat_path)["image_info"]
    return info[0, 0][0, 0][0].astype(np.float32).reshape(-1, 2)  # (N, 2) toạ độ (x, y)

rows = []
for split, items in [("train", train_all), ("test", test_items)]:
    for it in items:
        pts = load_points(it["gt"])
        img = cv2.imread(it["img"])
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape
        nn_d = cKDTree(pts).query(pts, k=2)[0][:, 1] if len(pts) > 1 else np.array([np.nan])
        it["count"] = len(pts)
        rows.append(dict(split=split, name=it["name"], count=len(pts), h=h, w=w,
                         brightness=float(gray.mean()), contrast=float(gray.std()),
                         is_gray=bool(np.allclose(img[..., 0], img[..., 1]) and np.allclose(img[..., 1], img[..., 2])),
                         nn_median=float(np.median(nn_d)), y_mean=float(pts[:, 1].mean() / h) if len(pts) else np.nan))
df = pd.DataFrame(rows)

def stats(items):
    c = np.array([it["count"] for it in items])
    return dict(n=len(c), total=int(c.sum()), min=int(c.min()), max=int(c.max()),
                mean=round(float(c.mean()), 1), median=float(np.median(c)))
data_stats = {"train": stats(train_all), "test": stats(test_items)}

summary = df.groupby("split")["count"].describe(percentiles=[.1, .25, .5, .75, .9]).round(1)
summary["tổng số người"] = df.groupby("split")["count"].sum()
display(summary)
print("Kích thước ảnh:", df.groupby(["h", "w"]).size().to_dict())
print("Số ảnh xám (grayscale):", int(df.is_gray.sum()))

### 3.2. Ảnh mẫu từ thưa đến dày

In [ ]:
tr = df[df.split == "train"].sort_values("count").reset_index(drop=True)
pick = tr.iloc[np.linspace(0, len(tr) - 1, 10).astype(int)]
by_name = {it["name"]: it for it in train_all}
fig, axes = plt.subplots(2, 5, figsize=(18, 6.5))
for ax, (_, r) in zip(axes.ravel(), pick.iterrows()):
    ax.imshow(cv2.cvtColor(cv2.imread(by_name[r["name"]]["img"]), cv2.COLOR_BGR2RGB))
    ax.set_title(f"{r['name']}: {r['count']} người", fontsize=11); ax.axis("off")
plt.suptitle("Ảnh train sắp xếp theo số người (phân vị 0% → 100%)", fontsize=13)
plt.tight_layout(); plt.savefig(f"{OUT}/gallery.png", dpi=110); plt.show()

### 3.3. Phân bố số người mỗi ảnh

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for split, color in [("train", "tab:blue"), ("test", "tab:orange")]:
    c = df[df.split == split]["count"]
    ax[0].hist(c, bins=30, alpha=.6, color=color, label=f"{split} (TB {c.mean():.0f})")
    ax[2].plot(np.sort(c), np.linspace(0, 1, len(c)), color=color, label=split)
ax[0].set_xlabel("Số người trong ảnh"); ax[0].set_ylabel("Số ảnh"); ax[0].legend(); ax[0].set_title("Histogram")
ax[1].boxplot([df[df.split == s]["count"] for s in ("train", "test")], labels=["train", "test"], showmeans=True)
ax[1].set_ylabel("Số người"); ax[1].set_title("Boxplot (tam giác = trung bình)")
ax[2].set_xlabel("Số người trong ảnh"); ax[2].set_ylabel("Tỉ lệ ảnh tích luỹ"); ax[2].legend(); ax[2].grid(alpha=.3)
ax[2].set_title("Hàm phân phối tích luỹ (CDF)")
plt.suptitle(f"Phân bố số người – ShanghaiTech Part {CFG['part']}", fontsize=13)
plt.tight_layout(); plt.savefig(f"{OUT}/hist_counts.png", dpi=150); plt.show()

from scipy.stats import ks_2samp
ks = ks_2samp(df[df.split == "train"]["count"], df[df.split == "test"]["count"])
print(f"Kiểm định Kolmogorov–Smirnov train vs test: p-value = {ks.pvalue:.3f} "
      f"({'không khác biệt đáng kể' if ks.pvalue > 0.05 else 'có khác biệt'} về phân bố số người)")

### 3.4. Nhãn điểm đầu người và vùng đông nhất

In [ ]:
def densest_window(pts, h, w, size=160):
    best, by, bx = -1, 0, 0
    for y in range(0, h - size + 1, 32):
        for x in range(0, w - size + 1, 32):
            n = int(((pts[:, 0] >= x) & (pts[:, 0] < x + size) & (pts[:, 1] >= y) & (pts[:, 1] < y + size)).sum())
            if n > best:
                best, by, bx = n, y, x
    return by, bx, best

picks = [tr.iloc[int(len(tr) * q)]["name"] for q in (0.1, 0.5, 0.99)]
fig, axes = plt.subplots(2, 3, figsize=(17, 9), gridspec_kw={"height_ratios": [3, 2]})
for j, name in enumerate(picks):
    it = by_name[name]
    img = cv2.cvtColor(cv2.imread(it["img"]), cv2.COLOR_BGR2RGB); pts = load_points(it["gt"]); h, w = img.shape[:2]
    y0, x0, n = densest_window(pts, h, w)
    axes[0, j].imshow(img); axes[0, j].scatter(pts[:, 0], pts[:, 1], s=5, c="red")
    axes[0, j].add_patch(plt.Rectangle((x0, y0), 160, 160, fill=False, ec="yellow", lw=2))
    axes[0, j].set_title(f"{name}: {len(pts)} người"); axes[0, j].axis("off")
    m = (pts[:, 0] >= x0) & (pts[:, 0] < x0 + 160) & (pts[:, 1] >= y0) & (pts[:, 1] < y0 + 160)
    axes[1, j].imshow(img[y0:y0 + 160, x0:x0 + 160]); axes[1, j].scatter(pts[m, 0] - x0, pts[m, 1] - y0, s=25, c="red")
    axes[1, j].set_title(f"Phóng to ô 160×160 px đông nhất: {n} người"); axes[1, j].axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/sample_points.png", dpi=120); plt.show()

### 3.5. Phân bố không gian của người trong khung hình

In [ ]:
xs, ys = [], []
for it in train_all:
    pts = load_points(it["gt"]); h, w = cv2.imread(it["img"]).shape[:2]
    xs.append(pts[:, 0] / w); ys.append(pts[:, 1] / h)
xs, ys = np.concatenate(xs), np.concatenate(ys)

fig, ax = plt.subplots(1, 2, figsize=(15, 4.8), gridspec_kw={"width_ratios": [3, 1]})
hm = ax[0].hist2d(xs, ys, bins=[64, 48], range=[[0, 1], [0, 1]], cmap="inferno")
ax[0].invert_yaxis(); ax[0].set_xlabel("x (tỉ lệ chiều rộng)"); ax[0].set_ylabel("y (tỉ lệ chiều cao)")
ax[0].set_title(f"Vị trí của {len(xs):,} đầu người trên toàn tập train"); plt.colorbar(hm[3], ax=ax[0], label="số đầu người")
ax[1].hist(ys, bins=40, orientation="horizontal", color="tab:purple"); ax[1].invert_yaxis()
ax[1].set_xlabel("Số đầu người"); ax[1].set_title("Phân bố theo chiều dọc")
plt.tight_layout(); plt.savefig(f"{OUT}/spatial_heatmap.png", dpi=150); plt.show()
print(f"{(ys < 0.2).mean() * 100:.1f}% đầu người nằm ở 20% phía trên ảnh, {(ys > 0.8).mean() * 100:.1f}% ở 20% phía dưới")

### 3.6. Hiệu ứng phối cảnh: khoảng cách tới người gần nhất theo vị trí dọc

Camera nhìn chéo xuống nên người ở xa (phía trên ảnh) trông nhỏ hơn và sát nhau hơn. Khoảng cách tới đầu người gần nhất là thước đo gián tiếp kích thước đầu người — căn cứ để chọn σ của nhân Gauss.

In [ ]:
yy, dd = [], []
for it in train_all:
    pts = load_points(it["gt"]); h = cv2.imread(it["img"]).shape[0]
    if len(pts) > 1:
        d = cKDTree(pts).query(pts, k=2)[0][:, 1]
        yy.append(pts[:, 1] / h); dd.append(d)
yy, dd = np.concatenate(yy), np.concatenate(dd)

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].hist(np.clip(dd, 0, 150), bins=60, color="tab:green")
for v, lab in [(np.median(dd), "trung vị"), (CFG["sigma"], "σ đang dùng")]:
    ax[0].axvline(v, ls="--", c="k" if lab == "trung vị" else "r", label=f"{lab} = {v:.1f} px")
ax[0].set_xlabel("Khoảng cách tới đầu người gần nhất (px)"); ax[0].set_ylabel("Số đầu người"); ax[0].legend()
ax[0].set_title("Phân bố khoảng cách láng giềng gần nhất")
ax[1].hexbin(yy, np.clip(dd, 0, 150), gridsize=40, cmap="viridis", mincnt=1, bins="log")
bins = np.linspace(0, 1, 11); mid = (bins[:-1] + bins[1:]) / 2
med = [np.median(dd[(yy >= a) & (yy < b)]) if ((yy >= a) & (yy < b)).any() else np.nan for a, b in zip(bins[:-1], bins[1:])]
ax[1].plot(mid, med, "r-o", label="trung vị theo dải")
ax[1].set_xlabel("y (0 = trên cùng, 1 = dưới cùng)"); ax[1].set_ylabel("Khoảng cách gần nhất (px)"); ax[1].legend()
ax[1].set_title("Khoảng cách gần nhất theo vị trí dọc (hiệu ứng phối cảnh)")
plt.tight_layout(); plt.savefig(f"{OUT}/perspective_nn.png", dpi=150); plt.show()
print("Trung vị khoảng cách theo dải y:", np.round(med, 1))

### 3.7. Mật độ cục bộ: số người trong mỗi ô 128×128 px

In [ ]:
CELL = 128
cells, peak = [], []
for it in train_all:
    pts = load_points(it["gt"]); h, w = cv2.imread(it["img"]).shape[:2]
    H, _, _ = np.histogram2d(pts[:, 1], pts[:, 0], bins=[np.arange(0, h + 1, CELL), np.arange(0, w + 1, CELL)])
    cells.append(H.ravel()); peak.append(H.max())
cells = np.concatenate(cells)

fig, ax = plt.subplots(1, 2, figsize=(15, 4.3))
ax[0].hist(cells, bins=np.arange(0, cells.max() + 2) - .5, color="tab:red", log=True)
ax[0].set_xlabel(f"Số người trong một ô {CELL}×{CELL} px"); ax[0].set_ylabel("Số ô (thang log)")
ax[0].set_title(f"{(cells == 0).mean() * 100:.0f}% ô trống – phần lớn bản đồ mật độ bằng 0")
c = np.array([it["count"] for it in train_all])
ax[1].scatter(c, peak, s=10, alpha=.6); ax[1].set_xlabel("Tổng số người trong ảnh"); ax[1].set_ylabel("Số người ở ô đông nhất")
ax[1].set_title("Tổng số người và điểm nóng cục bộ của từng ảnh")
plt.tight_layout(); plt.savefig(f"{OUT}/local_density.png", dpi=150); plt.show()

### 3.8. Đặc trưng ảnh: độ sáng, độ tương phản

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for split, color in [("train", "tab:blue"), ("test", "tab:orange")]:
    s = df[df.split == split]
    ax[0].hist(s.brightness, bins=30, alpha=.6, color=color, label=split)
    ax[1].hist(s.contrast, bins=30, alpha=.6, color=color, label=split)
    ax[2].scatter(s.brightness, s["count"], s=10, alpha=.6, color=color, label=split)
ax[0].set_xlabel("Độ sáng trung bình (0–255)"); ax[0].set_title("Độ sáng"); ax[0].legend()
ax[1].set_xlabel("Độ lệch chuẩn mức xám"); ax[1].set_title("Độ tương phản"); ax[1].legend()
ax[2].set_xlabel("Độ sáng trung bình"); ax[2].set_ylabel("Số người"); ax[2].set_title("Độ sáng và số người"); ax[2].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/image_props.png", dpi=150); plt.show()
print(f"Độ sáng trung bình mỗi ảnh: min {df.brightness.min():.0f}, trung vị {df.brightness.median():.0f}, max {df.brightness.max():.0f} "
      f"| ảnh tối (độ sáng < 60): {(df.brightness < 60).sum()} ảnh")

### 3.9. Dữ liệu trải qua những mức cảnh báo nào?

Giả định mỗi ảnh phủ diện tích $A = 100\,m^2$ (chỉ để minh hoạ), mật độ $\rho = C/A$ được xếp vào 4 mức theo ngưỡng đám đông di chuyển (Vàng ≥ 2, Cam ≥ 3, Đỏ ≥ 4 người/m²).

In [ ]:
AREA_EDA = 100.0
lv_names = ["Xanh", "Vàng", "Cam", "Đỏ"]; lv_colors = ["#2e7d32", "#f9a825", "#ef6c00", "#c62828"]
df["level"] = df["count"].apply(lambda c: int(sum(c / AREA_EDA >= t for t in (2, 3, 4))))
tab = df.pivot_table(index="level", columns="split", values="name", aggfunc="count").reindex(range(4)).fillna(0).astype(int)
tab.index = lv_names
display(tab)
ax = tab.plot.bar(figsize=(7, 3.8), color=["tab:blue", "tab:orange"], rot=0)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Số ảnh"); ax.set_title(f"Số ảnh theo mức cảnh báo (giả định A = {AREA_EDA:.0f} m²)")
plt.tight_layout(); plt.savefig(f"{OUT}/level_distribution.png", dpi=150); plt.show()

### Tóm tắt EDA

Ô dưới in ra các con số chính để đưa vào báo cáo (mục 3.2 của báo cáo). Những điểm cần đọc khi xem các hình trên:

- **Số người lệch phải mạnh** (nhiều ảnh thưa, ít ảnh rất đông) → MAE bị chi phối bởi các ảnh đông; cần xem thêm sai số theo nhóm mật độ ở mục 8.
- **Phối cảnh rõ** (3.6) → người phía trên ảnh nhỏ và sát nhau hơn; σ cố định là một xấp xỉ, σ thích ứng là hướng cải tiến.
- **Phần lớn ô ảnh trống** (3.7) → bản đồ mật độ rất thưa; hàm mất mát MSE bị chi phối bởi vùng nền, vì vậy nhân bản đồ với `density_scale` để gradient không quá nhỏ.
- **Độ sáng** (3.8): nếu gần như không có ảnh tối thì mô hình chưa được học điều kiện ban đêm → cần cẩn trọng khi áp dụng cho camera ban đêm; tăng cường độ sáng ±20% ở mục 5 chỉ bù được một phần.

In [ ]:
eda = {
    "so_anh": int(len(df)), "kich_thuoc": {f"{h}x{w}": int(n) for (h, w), n in df.groupby(["h", "w"]).size().items()},
    "nn_median_px": float(np.median(dd)), "nn_median_top_vs_bottom": [float(med[0]), float(med[-1])],
    "ti_le_o_trong_%": float((cells == 0).mean() * 100), "o_dong_nhat": int(cells.max()),
    "do_sang_min_max": [float(df.brightness.min()), float(df.brightness.max())],
    "ks_pvalue_train_test": float(ks.pvalue),
    "so_anh_theo_muc": {k: {s: int(v) for s, v in row.items()} for k, row in tab.iterrows()},
}
print(json.dumps(eda, indent=2, ensure_ascii=False))

## 4. Sinh bản đồ mật độ ground-truth

Mỗi đầu người tại $x_i$ được thay bằng một nhân Gauss chuẩn hoá (tổng = 1):

$$F(x) = \sum_i \delta(x - x_i) * G_{\sigma_i}(x)$$

- Part B: $\sigma$ cố định = 15 px. Part A: $\sigma_i = \beta \cdot \bar d_i$ (khoảng cách trung bình tới 3 người gần nhất, $\beta = 0.3$).
- Nhân bị cắt ở biên ảnh được chuẩn hoá lại, nên **tổng bản đồ = đúng số người**.
- Bản đồ được cộng gộp theo khối 8×8 về độ phân giải 1/8 — đúng kích thước đầu ra của CSRNet — và lưu `.npy`.

In [ ]:
DOWN = 8
_kernel_cache = {}

def gaussian_kernel(sigma):
    key = round(float(sigma), 1)
    if key not in _kernel_cache:
        r = max(1, int(3 * key)); ax = np.arange(-r, r + 1, dtype=np.float32)
        k = np.exp(-ax ** 2 / (2 * key ** 2)); k = np.outer(k, k)
        _kernel_cache[key] = (k / k.sum(), r)
    return _kernel_cache[key]

def make_density(points, h, w, sigma=15, adaptive=False, k=3, beta=0.3):
    den = np.zeros((h, w), np.float32)
    if len(points) == 0:
        return den
    if adaptive and len(points) > k:
        d, _ = cKDTree(points).query(points, k=k + 1)
        sigmas = np.clip(beta * d[:, 1:].mean(1), 1, 50)
    else:
        sigmas = np.full(len(points), sigma, np.float32)
    for (x, y), s in zip(points, sigmas):
        x = int(min(w - 1, max(0, x))); y = int(min(h - 1, max(0, y)))
        g, r = gaussian_kernel(s)
        x0, x1, y0, y1 = max(0, x - r), min(w, x + r + 1), max(0, y - r), min(h, y + r + 1)
        patch = g[y0 - (y - r): y1 - (y - r), x0 - (x - r): x1 - (x - r)]
        den[y0:y1, x0:x1] += patch / patch.sum()
    return den

def downsample_sum(den, f=DOWN):
    H, W = den.shape[0] // f * f, den.shape[1] // f * f
    return den[:H, :W].reshape(H // f, f, W // f, f).sum(axis=(1, 3))

DEN_DIR = f"density_part{CFG['part']}"; os.makedirs(DEN_DIR, exist_ok=True)
t0 = time.time(); errs = []
for it in train_all + test_items:
    it["den"] = os.path.join(DEN_DIR, it["name"] + ("_tr" if it in train_all else "_te") + ".npy")
    if os.path.exists(it["den"]):
        continue
    h, w = cv2.imread(it["img"]).shape[:2]
    pts = load_points(it["gt"])
    den = downsample_sum(make_density(pts, h, w, CFG["sigma"], CFG["adaptive"]))
    np.save(it["den"], den.astype(np.float32))
    errs.append(abs(den.sum() - len(pts)))
print(f"Xong sau {time.time() - t0:.0f}s; sai lệch tổng mật độ so với số điểm: {np.mean(errs) if errs else 0:.3f} người/ảnh (do điểm nằm ngoài vùng cắt bội số 8)")

In [ ]:
it = train_all[0]
img = cv2.cvtColor(cv2.imread(it["img"]), cv2.COLOR_BGR2RGB); den = np.load(it["den"])
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(img); ax[0].set_title(f"Ảnh gốc – {it['count']} người"); ax[0].axis("off")
ax[1].imshow(den, cmap="jet"); ax[1].set_title(f"Bản đồ mật độ 1/8 – tổng = {den.sum():.1f}"); ax[1].axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/density_gt.png", dpi=150); plt.show()

### So sánh các cách chọn σ

Cùng một vùng ảnh đông, bản đồ mật độ với σ nhỏ (4 px), σ đang dùng (15 px), σ lớn (30 px) và σ thích ứng. σ quá nhỏ cho các “gai” khó học; σ quá lớn làm nhoè, các đầu người sát nhau hoà vào nhau. Tổng (số người) của mọi bản đồ đều bằng nhau.

In [ ]:
it = by_name[tr.iloc[-1]["name"]]
img = cv2.cvtColor(cv2.imread(it["img"]), cv2.COLOR_BGR2RGB); pts = load_points(it["gt"]); h, w = img.shape[:2]
y0, x0, _ = densest_window(pts, h, w, size=min(256, h, w))
S = min(256, h, w)
variants = [("σ = 4 px", dict(sigma=4)), (f"σ = {CFG['sigma']} px (đang dùng)", dict(sigma=CFG["sigma"])),
            ("σ = 30 px", dict(sigma=30)), ("σ thích ứng (k=3, β=0.3)", dict(adaptive=True))]
fig, ax = plt.subplots(1, 5, figsize=(20, 5))
ax[0].imshow(img[y0:y0 + S, x0:x0 + S]); ax[0].set_title("Vùng ảnh đông nhất"); ax[0].axis("off")
for a, (title, kw) in zip(ax[1:], variants):
    den = make_density(pts, h, w, **kw)
    a.imshow(den[y0:y0 + S, x0:x0 + S], cmap="jet"); a.axis("off")
    a.set_title(f"{title}\ntổng cả ảnh = {den.sum():.1f} / {len(pts)}")
plt.tight_layout(rect=(0, 0, 1, 0.97)); plt.savefig(f"{OUT}/sigma_compare.png", dpi=130); plt.show()

## 5. Chia tập và Dataset

- 400 ảnh train gốc → **360 train / 40 val** (val dùng để chọn epoch tốt nhất, tập test chỉ dùng một lần ở cuối).
- Tăng cường: cắt ngẫu nhiên 512×512 (toạ độ bội số 8 để khớp bản đồ 1/8), lật ngang, thay đổi độ sáng nhẹ.
- Chuẩn hoá theo trung bình/độ lệch chuẩn ImageNet (vì frontend là VGG-16 huấn luyện trước).

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)

rng = random.Random(CFG["seed"]); idx = list(range(len(train_all))); rng.shuffle(idx)
n_val = int(len(idx) * CFG["val_ratio"])
val_items = [train_all[i] for i in idx[:n_val]]
train_items = [train_all[i] for i in idx[n_val:]]
print(len(train_items), "train /", len(val_items), "val /", len(test_items), "test")

class CrowdDataset(Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        it = self.items[i]
        img = cv2.cvtColor(cv2.imread(it["img"], cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        den = np.load(it["den"])
        H, W = den.shape[0] * DOWN, den.shape[1] * DOWN
        img = img[:H, :W]
        if self.train:
            ch, cw = min(CFG["crop"], H), min(CFG["crop"], W)
            y = random.randint(0, (H - ch) // DOWN) * DOWN
            x = random.randint(0, (W - cw) // DOWN) * DOWN
            img = img[y:y + ch, x:x + cw]
            den = den[y // DOWN:(y + ch) // DOWN, x // DOWN:(x + cw) // DOWN]
            if random.random() < 0.5:
                img, den = img[:, ::-1], den[:, ::-1]
            img = np.clip(img.astype(np.float32) * random.uniform(0.8, 1.2), 0, 255)
        x = (np.asarray(img, np.float32) / 255.0 - MEAN) / STD
        x = torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))
        d = torch.from_numpy(np.ascontiguousarray(den) * CFG["density_scale"]).unsqueeze(0)
        return x, d, float(it["count"])

nw = 2
train_loader = DataLoader(CrowdDataset(train_items, True), batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=nw, pin_memory=True, drop_last=True)
val_loader = DataLoader(CrowdDataset(val_items, False), batch_size=1, num_workers=nw)
test_loader = DataLoader(CrowdDataset(test_items, False), batch_size=1, num_workers=nw)

### Kiểm tra chia tập và xem trước dữ liệu sau tăng cường

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for name, items in [("train", train_items), ("val", val_items), ("test", test_items)]:
    c = np.sort([it["count"] for it in items]); ax.plot(c, np.linspace(0, 1, len(c)), label=f"{name} ({len(c)} ảnh, TB {c.mean():.0f})")
ax.set_xlabel("Số người trong ảnh"); ax.set_ylabel("Tỉ lệ tích luỹ"); ax.grid(alpha=.3); ax.legend()
ax.set_title("Phân bố số người của ba tập train / val / test")
plt.tight_layout(); plt.savefig(f"{OUT}/split_cdf.png", dpi=150); plt.show()

ds = CrowdDataset(train_items, True)
fig, axes = plt.subplots(2, 4, figsize=(18, 8.5))
for k, a in enumerate(axes.ravel()):
    x, d, _ = ds[k % 4]  # mỗi ảnh lấy 2 lần để thấy cắt/lật ngẫu nhiên khác nhau
    im = np.clip(x.numpy().transpose(1, 2, 0) * STD + MEAN, 0, 1)
    dm = cv2.resize(d[0].numpy(), (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST)
    a.imshow(im); a.imshow(dm, cmap="jet", alpha=.35); a.axis("off")
    a.set_title(f"{train_items[k % 4]['name']} – ô cắt có {d.sum().item() / CFG['density_scale']:.1f} người")
plt.suptitle("Mẫu huấn luyện sau tăng cường (cắt 512×512, lật ngang, đổi độ sáng) + bản đồ mật độ phủ lên", fontsize=13)
plt.tight_layout(); plt.savefig(f"{OUT}/augment_preview.png", dpi=110); plt.show()

## 6. Mô hình CSRNet

- **Frontend:** 10 lớp tích chập đầu của VGG-16 (3 lớp max-pool → đầu ra 1/8), khởi tạo từ trọng số ImageNet.
- **Backend:** 6 lớp tích chập 3×3 **giãn** (dilation = 2) với số kênh 512-512-512-256-128-64: mở rộng vùng tiếp nhận mà không giảm độ phân giải.
- **Đầu ra:** tích chập 1×1 → bản đồ mật độ 1 kênh. Số người = tổng bản đồ / `density_scale`.

In [ ]:
def make_layers(cfg, in_ch=3, dilation=1):
    layers = []
    for v in cfg:
        if v == "M":
            layers.append(nn.MaxPool2d(2, 2))
        else:
            layers += [nn.Conv2d(in_ch, v, 3, padding=dilation, dilation=dilation), nn.ReLU(inplace=True)]
            in_ch = v
    return nn.Sequential(*layers)

class CSRNet(nn.Module):
    def __init__(self, load_vgg=True):
        super().__init__()
        self.frontend = make_layers([64, 64, "M", 128, 128, "M", 256, 256, 256, "M", 512, 512, 512])
        self.backend = make_layers([512, 512, 512, 256, 128, 64], 512, dilation=2)
        self.output = nn.Conv2d(64, 1, 1)
        for m in list(self.backend.modules()) + [self.output]:
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight, std=0.01); nn.init.zeros_(m.bias)
        if load_vgg:
            import torchvision
            vgg = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1)
            self.frontend.load_state_dict(vgg.features[:23].state_dict())
    def forward(self, x):
        return self.output(self.backend(self.frontend(x)))

model = CSRNet().to(DEVICE)
print(f"Số tham số: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} triệu")
with torch.no_grad():
    print("Vào 1×3×768×1024 → ra", tuple(model(torch.zeros(1, 3, 768, 1024, device=DEVICE)).shape))

## 7. Huấn luyện

- Hàm mất mát: MSE (tổng theo điểm ảnh, chia cho batch) giữa bản đồ dự đoán và bản đồ ground-truth.
- Tối ưu Adam, lr = 1e-5, giảm dần theo cosine; huấn luyện hỗn hợp độ chính xác (AMP) để nhanh hơn trên GPU.
- Cứ `eval_every` epoch tính MAE trên tập val, lưu checkpoint có MAE val nhỏ nhất.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); preds, gts = [], []
    for x, _, c in loader:
        with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
            out = model(x.to(DEVICE))
        preds.append(out.float().sum().item() / CFG["density_scale"]); gts.append(float(c))
    p, g = np.array(preds), np.array(gts)
    return dict(mae=float(np.abs(p - g).mean()), rmse=float(np.sqrt(((p - g) ** 2).mean()))), p, g

opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"], eta_min=CFG["lr"] * 0.05)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
criterion = nn.MSELoss(reduction="sum")
BEST = f"{OUT}/csrnet_part{CFG['part']}.pth"
history, best_mae, t_start = [], float("inf"), time.time()

for epoch in range(1, CFG["epochs"] + 1):
    model.train(); run_loss = 0.0
    for x, d, _ in train_loader:
        x, d = x.to(DEVICE, non_blocking=True), d.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
            out = model(x)
        loss = criterion(out.float(), d) / x.size(0)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        run_loss += loss.item()
    sched.step()
    rec = {"epoch": epoch, "loss": run_loss / len(train_loader)}
    if epoch % CFG["eval_every"] == 0 or epoch == CFG["epochs"]:
        m, _, _ = evaluate(model, val_loader)
        rec.update(val_mae=m["mae"], val_rmse=m["rmse"])
        if m["mae"] < best_mae:
            best_mae = m["mae"]
            torch.save({"model": model.state_dict(), "density_scale": CFG["density_scale"], "cfg": CFG,
                        "epoch": epoch, "val_mae": m["mae"], "dataset": f"ShanghaiTech Part {CFG['part']}"}, BEST)
        print(f"Epoch {epoch:3d} | loss {rec['loss']:.3f} | val MAE {m['mae']:.2f} RMSE {m['rmse']:.2f} | best {best_mae:.2f} | {(time.time() - t_start) / 60:.1f} phút")
    history.append(rec)

json.dump(history, open(f"{OUT}/history.json", "w", encoding="utf-8"), indent=1)
print(f"Tổng thời gian huấn luyện: {(time.time() - t_start) / 60:.1f} phút")

In [ ]:
ep = [h["epoch"] for h in history]
ev = [h for h in history if "val_mae" in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(ep, [h["loss"] for h in history]); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss (MSE)"); ax[0].set_title("Hàm mất mát trên tập train")
ax[1].plot([h["epoch"] for h in ev], [h["val_mae"] for h in ev], label="MAE")
ax[1].plot([h["epoch"] for h in ev], [h["val_rmse"] for h in ev], label="RMSE")
ax[1].set_xlabel("Epoch"); ax[1].set_title("Sai số đếm trên tập val"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/training_curves.png", dpi=150); plt.show()

## 8. Đánh giá trên tập test

In [ ]:
ckpt = torch.load(BEST, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model"]); model.eval()
test_m, p, g = evaluate(model, test_loader)
print(f"Checkpoint epoch {ckpt['epoch']} (val MAE {ckpt['val_mae']:.2f})")
print(f"TEST  MAE = {test_m['mae']:.2f}   RMSE = {test_m['rmse']:.2f}   (CSRNet gốc trên Part B: 10.6 / 16.0)")

# Sai số theo nhóm mật độ
bins = [(0, 50), (50, 100), (100, 200), (200, 10000)]
by_bin = []
for lo, hi in bins:
    m = (g >= lo) & (g < hi)
    if m.any():
        by_bin.append({"nhom": f"{lo}–{hi if hi < 10000 else '∞'}", "so_anh": int(m.sum()),
                       "mae": float(np.abs(p[m] - g[m]).mean()), "sai_so_tuong_doi_%": float((np.abs(p[m] - g[m]) / g[m]).mean() * 100)})
for b in by_bin: print(b)

# Tốc độ suy luận trên ảnh 768×1024
x = torch.zeros(1, 3, 768, 1024, device=DEVICE)
with torch.no_grad(), torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
    for _ in range(5): model(x)
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(30): model(x)
    if DEVICE == "cuda": torch.cuda.synchronize()
fps = 30 / (time.time() - t0)
print(f"Tốc độ: {fps:.1f} ảnh/giây trên {DEVICE}")

results = {"dataset": ckpt["dataset"], "data_stats": data_stats, "eda": eda, "best_epoch": ckpt["epoch"], "val_mae": ckpt["val_mae"],
           "test_mae": test_m["mae"], "test_rmse": test_m["rmse"], "by_bin": by_bin, "fps_768x1024": fps,
           "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu", "cfg": CFG}
json.dump(results, open(f"{OUT}/results.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(g, p, s=10, alpha=.6); lim = max(g.max(), p.max()) * 1.05
plt.plot([0, lim], [0, lim], "r--", lw=1); plt.xlim(0, lim); plt.ylim(0, lim)
plt.xlabel("Số người thực tế"); plt.ylabel("Số người dự đoán")
plt.title(f"Tập test – MAE {test_m['mae']:.2f}, RMSE {test_m['rmse']:.2f}")
plt.tight_layout(); plt.savefig(f"{OUT}/scatter_test.png", dpi=150); plt.show()

show = [test_items[i] for i in np.argsort(g)[[10, len(g) // 2, -15, -1]]]
fig, ax = plt.subplots(len(show), 3, figsize=(13, 3.2 * len(show)))
for r, it in enumerate(show):
    x, d, c = test_loader.dataset[test_items.index(it)]
    with torch.no_grad():
        out = model(x[None].to(DEVICE))[0, 0].float().cpu().numpy() / CFG["density_scale"]
    img = cv2.cvtColor(cv2.imread(it["img"]), cv2.COLOR_BGR2RGB)
    ax[r, 0].imshow(img); ax[r, 0].set_title(f"Ảnh test {it['name']}")
    ax[r, 1].imshow(d[0].numpy() / CFG["density_scale"], cmap="jet"); ax[r, 1].set_title(f"Ground truth: {c:.0f} người")
    ax[r, 2].imshow(np.clip(out, 0, None), cmap="jet"); ax[r, 2].set_title(f"CSRNet dự đoán: {out.sum():.1f} người")
    for a in ax[r]: a.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/test_samples.png", dpi=130); plt.show()

## 9. Thử luật cảnh báo mật độ

Mật độ $\rho = C / A$ với $A$ là diện tích thực (m²) của vùng camera nhìn thấy. ShanghaiTech không có thông tin hiệu chỉnh camera, nên ở đây **giả định** mỗi ảnh phủ $A = 100\,m^2$ chỉ để minh hoạ cách các mức Xanh / Vàng / Cam / Đỏ được gán. Trong app demo, người vận hành nhập $A$ cho từng camera.

In [ ]:
AREA = 100.0
THR = (2.0, 3.0, 4.0)  # đám đông di chuyển: Vàng ≥ 2, Cam ≥ 3, Đỏ ≥ 4 người/m²
NAMES = ["Xanh", "Vàng", "Cam", "Đỏ"]; COLORS = ["#2e7d32", "#f9a825", "#ef6c00", "#c62828"]
lvl = lambda r: int(sum(r >= t for t in THR))
order = np.argsort(g)
rho_p, rho_g = p[order] / AREA, g[order] / AREA
lp, lg = np.array([lvl(r) for r in rho_p]), np.array([lvl(r) for r in rho_g])
print(f"Mức cảnh báo từ dự đoán trùng với mức từ ground truth: {(lp == lg).mean() * 100:.1f}% ảnh")
print("Số ảnh lệch 2 mức trở lên:", int((np.abs(lp - lg) >= 2).sum()))
results["alert_agreement"] = float((lp == lg).mean() * 100)
json.dump(results, open(f"{OUT}/results.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)

plt.figure(figsize=(10, 3.8))
edges = [0, *THR, max(rho_p.max(), rho_g.max()) * 1.1]
for i in range(4):
    plt.axhspan(edges[i], edges[i + 1], color=COLORS[i], alpha=.12, label=f"Mức {NAMES[i]}")
plt.plot(rho_g, "k-", lw=1, label="Mật độ thực (GT)")
plt.scatter(range(len(rho_p)), rho_p, s=6, c=[COLORS[l] for l in lp], label="Mật độ dự đoán")
plt.xlabel("Ảnh test (sắp xếp theo số người)"); plt.ylabel("người/m²"); plt.legend(fontsize=8, ncol=3)
plt.title(f"Mức cảnh báo theo mật độ (giả định A = {AREA:.0f} m²)")
plt.tight_layout(); plt.savefig(f"{OUT}/alert_levels.png", dpi=150); plt.show()

## 10. Xuất kết quả

Tải `outputs.zip` về máy. Chép `csrnet_partB.pth` vào thư mục `code/weights/` để chạy app demo; các file `.png` và `results.json` dùng để điền Chương 4 của báo cáo.

In [ ]:
shutil.make_archive("outputs", "zip", OUT)
print(os.listdir(OUT))
try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Không chạy trên Colab: lấy file outputs.zip trong thư mục hiện tại.")